# PAIRS TRADING ANALYSIS
Bank of Baroda vs Union Bank of India

In [1]:
# pip install yfinance pandas numpy statsmodels scipy matplotlib
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from datetime import datetime

In [2]:
HDFCBANK   = "HDFCBANK.NS"
ICICIBANK  = "ICICIBANK.NS"
KOTAKBANK  = "KOTAKBANK.NS"
SBIN       = "SBIN.NS"
AXISBANK   = "AXISBANK.NS"
INDUSINDBK = "INDUSINDBK.NS"
FEDERALBNK = "FEDERALBNK.NS"
IDFCFIRSTB = "IDFCFIRSTB.NS"
PNB        = "PNB.NS"
BOB        = "BANKBARODA.NS"
CANBK      = "CANBK.NS"
UNION      = "UNIONBANK.NS"
AUBANK     = "AUBANK.NS"
YESBANK    = "YESBANK.NS"
stock_names = {HDFCBANK:"HDFC Bank",ICICIBANK:"ICICI Bank",KOTAKBANK:"Kotak Mahindra Bank",SBIN:"State Bank of India",AXISBANK:"Axis Bank",INDUSINDBK:"IndusInd Bank",FEDERALBNK:"Federal Bank",IDFCFIRSTB:"IDFC First Bank",PNB:"Punjab National Bank",BOB:"Bank of Baroda",CANBK:"Canara Bank",UNION:"Union Bank of India",AUBANK:"AU Small Finance Bank",YESBANK:"Yes Bank"}

In [3]:
end_date = datetime.today()
start_date = end_date - pd.DateOffset(years=5)
print("Downloading 5 years of stock data...")
raw = yf.download(list(stock_names.keys()), start=start_date, end=end_date, auto_adjust=True, progress=False)
close = raw["Close"].copy().rename(columns=stock_names)
data = close.dropna()
data = data[~data.index.duplicated(keep="first")].sort_index()

In [4]:
def correlation(ticker1, ticker2, data, threshold):
    c = data[ticker1].corr(data[ticker2])
    print("\n" + "=" * 60)
    print(f"CORRELATION {ticker1}-{ticker2}")
    print("=" * 60)
    print(f"Correlation = {c:.4f}")
    print(f"Correlation % = {c*100:.2f}%")
    if c >= threshold:
        print("PASS: Correlation >= 75%")
        return True
    print("FAIL: Correlation below 75%")
    return False

In [5]:
correlated_tickers = []
for ticker1 in stock_names.values():
    for ticker2 in stock_names.values():
        if ticker1 != ticker2:
            if correlation(ticker1, ticker2, data, 0.75):
                correlated_tickers.append([ticker1, ticker2])


CORRELATION HDFC Bank-ICICI Bank
Correlation = 0.7752
Correlation % = 77.52%
PASS: Correlation >= 75%

CORRELATION HDFC Bank-Kotak Mahindra Bank
Correlation = 0.6811
Correlation % = 68.11%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-State Bank of India
Correlation = 0.5827
Correlation % = 58.27%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-Axis Bank
Correlation = 0.6324
Correlation % = 63.24%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-IndusInd Bank
Correlation = -0.3302
Correlation % = -33.02%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-Federal Bank
Correlation = 0.5172
Correlation % = 51.72%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-IDFC First Bank
Correlation = 0.4273
Correlation % = 42.73%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-Punjab National Bank
Correlation = 0.5919
Correlation % = 59.19%
FAIL: Correlation below 75%

CORRELATION HDFC Bank-Bank of Baroda
Correlation = 0.6407
Correlation % = 64.07%
FAIL: Correlation below 75%

CORRE

In [6]:
def run_regression(y, x, y_name, x_name):
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    residuals = model.resid
    return {
        "model": model,
        "Y": y_name,
        "X": x_name,
        "intercept": model.params["const"],
        "slope": model.params[x.name],
        "intercept_se": model.bse["const"],
        "residual_std": residuals.std(ddof=1),
        "residuals": residuals,
    }

In [7]:
seen = set()
stripped = []
for pair in correlated_tickers:
    key = frozenset(pair)
    if key not in seen:
        seen.add(key)
        stripped.append(pair)

pairs = []
for ticker1, ticker2 in stripped:
    reg1 = run_regression(y=data[ticker1], x=data[ticker2], y_name=ticker1, x_name=ticker2)
    reg2 = run_regression(y=data[ticker2], x=data[ticker1], y_name=ticker2, x_name=ticker1)
    selected = reg1 if (reg1["intercept_se"]/reg1["residual_std"]) < (reg2["intercept_se"]/reg2["residual_std"]) else reg2
    print(f"\nSELECTED: Y={selected['Y']}, X={selected['X']}, ResidualSTD={selected['residual_std']:.6f}")
    adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
    print(f"ADF Stat: {adf_stat:.6f} | p-value: {p_value:.6f}")
    if p_value <= 0.05:
        print("PASS — pair CONFIRMED")
        pairs.append([ticker1, ticker2])
    else:
        print("FAIL — pair NOT confirmed")

print("\nPAIRS:")
for a, b in pairs:
    print(f"• {a} ↔ {b}")


SELECTED: Y=HDFC Bank, X=ICICI Bank, ResidualSTD=60.237375
ADF Stat: -1.815582 | p-value: 0.372754
FAIL — pair NOT confirmed

SELECTED: Y=ICICI Bank, X=State Bank of India, ResidualSTD=106.609983
ADF Stat: -2.053105 | p-value: 0.263784
FAIL — pair NOT confirmed

SELECTED: Y=Axis Bank, X=ICICI Bank, ResidualSTD=89.921233
ADF Stat: -2.544105 | p-value: 0.105108
FAIL — pair NOT confirmed

SELECTED: Y=ICICI Bank, X=Federal Bank, ResidualSTD=123.873742
ADF Stat: -1.386940 | p-value: 0.588521
FAIL — pair NOT confirmed

SELECTED: Y=ICICI Bank, X=Punjab National Bank, ResidualSTD=118.733023
ADF Stat: -1.958999 | p-value: 0.304835
FAIL — pair NOT confirmed

SELECTED: Y=ICICI Bank, X=Bank of Baroda, ResidualSTD=111.480912
ADF Stat: -1.864877 | p-value: 0.348760
FAIL — pair NOT confirmed

SELECTED: Y=ICICI Bank, X=Canara Bank, ResidualSTD=106.639177


/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adful

ADF Stat: -2.098075 | p-value: 0.245289
FAIL — pair NOT confirmed

SELECTED: Y=ICICI Bank, X=Union Bank of India, ResidualSTD=112.875069
ADF Stat: -1.969701 | p-value: 0.300014
FAIL — pair NOT confirmed

SELECTED: Y=Axis Bank, X=State Bank of India, ResidualSTD=77.860411
ADF Stat: -3.087234 | p-value: 0.027512
PASS — pair CONFIRMED

SELECTED: Y=State Bank of India, X=Federal Bank, ResidualSTD=59.597503
ADF Stat: -1.831191 | p-value: 0.365090
FAIL — pair NOT confirmed

SELECTED: Y=State Bank of India, X=Punjab National Bank, ResidualSTD=93.895724
ADF Stat: -1.416383 | p-value: 0.574397
FAIL — pair NOT confirmed

SELECTED: Y=State Bank of India, X=Bank of Baroda, ResidualSTD=83.497547
ADF Stat: -1.107090 | p-value: 0.712236
FAIL — pair NOT confirmed

SELECTED: Y=State Bank of India, X=Canara Bank, ResidualSTD=52.477154


/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adful

ADF Stat: -2.525904 | p-value: 0.109298
FAIL — pair NOT confirmed

SELECTED: Y=State Bank of India, X=Union Bank of India, ResidualSTD=64.676018
ADF Stat: -2.588199 | p-value: 0.095461
FAIL — pair NOT confirmed

SELECTED: Y=Axis Bank, X=Federal Bank, ResidualSTD=97.536437
ADF Stat: -2.235522 | p-value: 0.193606
FAIL — pair NOT confirmed

SELECTED: Y=Axis Bank, X=IDFC First Bank, ResidualSTD=133.500571
ADF Stat: -1.864414 | p-value: 0.348983
FAIL — pair NOT confirmed

SELECTED: Y=Axis Bank, X=Punjab National Bank, ResidualSTD=76.824884
ADF Stat: -3.519520 | p-value: 0.007500
PASS — pair CONFIRMED

SELECTED: Y=Axis Bank, X=Bank of Baroda, ResidualSTD=57.600611
ADF Stat: -4.635335 | p-value: 0.000111
PASS — pair CONFIRMED

SELECTED: Y=Axis Bank, X=Canara Bank, ResidualSTD=65.300346
ADF Stat: -3.991397 | p-value: 0.001456
PASS — pair CONFIRMED


...

/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adful

ADF Stat: -2.124110 | p-value: 0.234931
FAIL — pair NOT confirmed

SELECTED: Y=Bank of Baroda, X=Union Bank of India, ResidualSTD=19.170111
ADF Stat: -1.445974 | p-value: 0.560048
FAIL — pair NOT confirmed

SELECTED: Y=Yes Bank, X=Bank of Baroda, ResidualSTD=1.992860
ADF Stat: -3.042092 | p-value: 0.031146
PASS — pair CONFIRMED

SELECTED: Y=Canara Bank, X=Union Bank of India, ResidualSTD=8.215810
ADF Stat: -2.484443 | p-value: 0.119306
FAIL — pair NOT confirmed

SELECTED: Y=Yes Bank, X=Canara Bank, ResidualSTD=2.227221
ADF Stat: -2.725588 | p-value: 0.069714
FAIL — pair NOT confirmed

SELECTED: Y=Yes Bank, X=Union Bank of India, ResidualSTD=2.204398


/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adful

ADF Stat: -2.900854 | p-value: 0.045249
PASS — pair CONFIRMED

PAIRS:
• State Bank of India ↔ Axis Bank
• Axis Bank ↔ Punjab National Bank
• Axis Bank ↔ Bank of Baroda
• Axis Bank ↔ Canara Bank
• Axis Bank ↔ Union Bank of India
• Punjab National Bank ↔ Yes Bank
• Bank of Baroda ↔ Yes Bank
• Union Bank of India ↔ Yes Bank


/tmp/ipykernel_134912/1860274312.py:15: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, p_value, *_ = adfuller(selected["residuals"], autolag="AIC")
